## 🇺🇦 Wartime Civilian Harms and Adolescent Trauma in Ukraine

_WIP - NOT FOR DISTRIBUTION_

**Preregistration and STROBE checklist in progress**

*Provided in the spirit of open science. This notebook offers a "one-click" replication wrangling and building the _oblast_ (region) and _raion_ (district) Bellingcat OSINT [Civilian Harm in Ukraine](https://ukraine.bellingcat.com/) geocoded event-level data $\rightarrow$ Ukraine Longitudinal Study (ULS) and (non-replicable) individual-level _Ukraine Longitudinal Survey_ (ULS) 1:$n$ merge. To protect the safety and anonymity of the child-adolescent ULS respondents, those data are private and cannot be released. 

⛏️ `uls_scratchpad.ipynb`<br>
Simone J. Skeen x Claude Code (06-18-2026)

1. [Prepare](#1-prepare)
2. [Import + transform](#2-import--transform-bellingcat-osint-civilian-harm-in-ukraine)


### 1. Prepare
Imports requisite packages; customizes outputs.
***
**Dependencies:** Install via `pip install -r requirements.txt` from project root before running.

In [ ]:
%%capture

%pip install -r ../../requirements.txt

In [ ]:
# Standard library
import json
import os
import re
import sys
import urllib.request
import warnings
from datetime import datetime
from pathlib import Path
from time import sleep

# Add src to path for local imports
sys.path.insert(0, str(Path.cwd().parent))

# Third-party
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

from dotenv import load_dotenv
from geopy.geocoders import Nominatim
from IPython.core.interactiveshell import InteractiveShell
from tqdm.notebook import tqdm

# Local
from mappings import ADMIN_UNIT_TO_OBLAST, RAION_UA_TO_EN

In [ ]:
# Env variables
load_dotenv()
BELLINGCAT_API_URL = os.getenv('BELLINGCAT_API_URL')

# Output preferences
InteractiveShell.ast_node_interactivity = 'all'

pd.options.mode.copy_on_write = True
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

for category in (FutureWarning, UserWarning):
    warnings.simplefilter(action='ignore', category=category)

In [ ]:
%%script false --no-raise-error

# Project directory structure
.
└── ukraine_longitudinal_survey/
    ├── config
    ├── data/
    │   ├── raw/
    │   │   ├── level_1
    │   │   └── level_2
    │   └── processed
    ├── src/
    │   └── notebooks
    └── outputs/
        └── figures

In [ ]:
# Set working directory to project root; define data paths
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('../..')
elif os.path.basename(os.getcwd()) == 'src':
    os.chdir('..')

# Inputs subdirectories
DATA_RAW = 'data/raw'
DATA_PROC = 'data/processed'
DATA_LVL1 = f'{DATA_RAW}/level_1'   ### ULS child-adolescent survey data; not for public use
DATA_LVL2 = f'{DATA_RAW}/level_2'   ### Bellingcat geotagged OSINT data

In [ ]:
# Bellingcat data source
BELLINGCAT_CSV = 'ukr-civharm-2026-01-09.csv'

# ULS merge params
SURVEY_START_DATE = '2025-04-08' ### earliest observation in ULS survey
DATE_FORMAT_INPUT = '%m/%d/%Y'   ### format in source data (if .CSV fallback)
DATE_FORMAT_ISO = '%Y-%m-%d'     ### ISO 8601 for internal use

# Ukrposhta postcode directory (data.gov.ua)
POSTCODE_7Z = 'zvit-dlia-miu-perelik-poshtovikh-indeksiv-ta-viddilen_08-08-2025-csv.7z'

# Geocoding
NOMINATIM_USER_AGENT = 'ukraine_postcode_geocoder'
NOMINATIM_DELAY_SEC = 1  ### 1-second delay; ensures rate limit compliance

In [ ]:
%%script false --no-raise-error
#############################################################################################

# Fetch routinely updated .JSON from Bellingcat API endpoint

### docs: https://github.com/bellingcat/ukraine-timemap

def fetch_bellingcat_json(url):
    """
    Fetches Bellingcat civilian harm data from API endpoint.
    Returns list of event dictionaries.
    """
    with urllib.request.urlopen(url) as response:
        data = json.loads(response.read().decode('utf-8'))
    return data

def convert_to_csv_format(events):
    """
    Converts Bellingcat API JSON to CSV format matching ukr-civharm-*.csv structure.
    
    JSON format: id, date (YYYY-MM-DD), latitude, longitude, location, 
                 description, sources (array), impact (array), weapon_system (array)
    CSV format:  id, date (MM/DD/YYYY), latitude, longitude, location,
                 description, sources (comma-sep), associations (formatted string)
    """
    rows = []
    for event in events:
        # Convert date: YYYY-MM-DD → MM/DD/YYYY
        date_iso = event.get('date', '')
        try:
            date_obj = datetime.strptime(date_iso, '%Y-%m-%d')
            date_formatted = date_obj.strftime('%m/%d/%Y')
        except ValueError:
            date_formatted = date_iso
        
        # Join sources array
        sources = event.get('sources', [])
        sources_str = ','.join(sources) if sources else ''
        
        # Build `associations` string from `impact` & `weapon_system`
        associations_parts = []
        for impact in event.get('impact', []):
            associations_parts.append(f'Type of area affected={impact}')
        for weapon in event.get('weapon_system', []):
            associations_parts.append(f'Weapon System={weapon}')
        associations_str = ','.join(associations_parts) if associations_parts else ''
        
        rows.append({
            'id': event.get('id', ''),
            'date': date_formatted,
            'latitude': event.get('latitude', ''),
            'longitude': event.get('longitude', ''),
            'location': event.get('location', '').strip(),
            'description': event.get('description', '').strip(),
            'sources': sources_str,
            'associations': associations_str,
        })
    
    return pd.DataFrame(rows)

# Fetch
print(f"Fetching data from Bellingcat API...")
d_lvl2_raw_json = fetch_bellingcat_json(BELLINGCAT_API_URL)

# Save raw .JSON 
json_path = f'{DATA_LVL2}/ukr-civharm-{datetime.now().strftime('%Y-%m-%d')}.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(d_lvl2_raw_json, f, ensure_ascii=False, indent=2)
print(f"Raw JSON saved to: {json_path}")

# Convert & save to .CSV
d_lvl2_raw_csv = convert_to_csv_format(d_lvl2_raw_json)
csv_path = f'{DATA_LVL2}/ukr-civharm-{datetime.now().strftime('%Y-%m-%d')}.csv'
d_lvl2_raw_csv.to_csv(csv_path, index=False)

print(f"Fetched {len(d_lvl2_raw_csv):,} events")
print(f"Saved to: {csv_path}")
d_lvl2_raw_csv.head(3)
#############################################################################################

### 2. Import + transform: Bellingcat OSINT Civilian Harm in Ukraine
Imports, cleans, describes level-2 aggregate conflict data. Acquired via .csv export: https://ukraine.bellingcat.com/.

In [ ]:
#############################################################################################
d = pd.read_csv(f'data/ukr-civharm-2026-01-09.csv')
d.head(5)
#############################################################################################

In [ ]:
# Dupe raw for processing
#d = d_lvl2_raw_csv.copy()

# Add ascending numerical index
d['index'] = range(len(d))
d = d.set_index('index')

# Drop imprecise OS location col
d = d.drop(
    'location', 
    axis = 1, 
    errors = 'ignore',
    )

# Restrict to obs on or before ULS start date
d['date'] = pd.to_datetime(
    d['date'], 
    format = DATE_FORMAT_INPUT,
    errors = 'coerce',    
    )

uls_startdate = pd.to_datetime(SURVEY_START_DATE)
d = d[d['date'] <= uls_startdate]

# Inspect & verify
d.shape
d.info()
d.head(2)
d.tail(2)

In [ ]:
# Dummy code: area type affected & weapon system

### Creates binary indicators for all `associations` values

# === TYPE OF AREA AFFECTED ===
area_types = {
    'admn': 'Administrative',           
    'cmrc': 'Commercial',
    'cltr': 'Cultural',
    'food': 'Food/Food Infrastructure',
    'hlth': 'Healthcare',
    'hmnt': 'Humanitarian',
    'indl': 'Industrial',
    'mltr': 'Military',
    'rlgs': 'Religious',
    'rsdn': 'Residential',
    'trsp': 'Roads/Highways/Transport',
    'schl': 'School or childcare',
    'undf': 'Undefined',
    }

for var, label in area_types.items():
    d[var] = d['associations'].str.contains(
        rf'Type of area affected={re.escape(label)}',
        case=False,
        na=False,
        regex=True,
    ).astype(int)

# === WEAPON SYSTEM ===
weapon_systems = {
    'arst': 'Air strike',
    'amsl': 'Anti-air missile',
    'blsc': 'Ballistic missile',
    'clst': 'Cluster munitions',
    'crsm': 'Cruise missile',
    'mrtr': 'HE artillery inc mortars',
    'rckt': 'HE rocket artillery',
    'tube': 'HE tube artillery',
    'incd': 'Incendiary munitions',
    'mine': 'Land mines',
    'ltrm': 'Loitering munition',
    'none': 'None',
    'smll': 'Small arms',
    'thrm': 'Thermobaric munition',
    'unkn': 'Unknown',
    'vhcl': 'Vehicle mounted weapon',
    }

for var, label in weapon_systems.items():
    d[var] = d['associations'].str.contains(
        rf'Weapon System={re.escape(label)}',
        case=False,
        na=False,
        regex=True,
    ).astype(int)

# Verify counts

print("=== TYPE OF AREA AFFECTED ===")
for var, label in area_types.items():
    print(f"  {var} ({label}): {d[var].sum()}")

print("\n=== WEAPON SYSTEM ===")
for var, label in weapon_systems.items():
    print(f"  {var} ({label}): {d[var].sum()}")

In [ ]:
#############################################################################################
d.head(5)
#############################################################################################

#### 2a. Reverse geocode: latitude / longitude $\rightarrow$ UA postcode
Reverse geocodes event coordinates to Ukrainian postcodes via Nominatim API.

In [ ]:
#############################################################################################
# Restrict to n = 100 for geocoding tests
#d = d.iloc[:100]
#d.info()
#############################################################################################

In [ ]:
# Convert latitude/longitude coordinates → Ukrainian postcodes via Nominatim API
geolocator = Nominatim(user_agent = NOMINATIM_USER_AGENT)

def get_postcode(lat, lon):
    """
    Reverse geocode latitude/longitude to get postcode.
    Returns None if postcode not found.
    """
    try:
        location = geolocator.reverse(f"{lat}, {lon}", language = 'en')
        if location and location.raw.get('address'):
            postcode = location.raw['address'].get('postcode')
            return postcode
        return None
    except Exception as e:
        print(f"Error geocoding ({lat}, {lon}): {e}")
        return None

# Apply geocoding to each row with rate-limited delay
postcodes = []

for idx, row in tqdm(d.iterrows(), total=len(d), desc="Geocoding postcodes"):
    lat = row['latitude']
    lon = row['longitude']
    
    postcode = get_postcode(lat, lon)
    postcodes.append(postcode)
    
    sleep(NOMINATIM_DELAY_SEC)

d['postcode'] = postcodes

print(f"\nGeocoding complete!")
print(f"Postcodes found: {d['postcode'].notna().sum()}/{len(d)}")
print(f"\nSample results:")
print(d[['latitude', 'longitude', 'postcode']].head(10))

In [ ]:
# Enumerate unique postcodes in d
def count_unique_postcodes(df, col = 'postcode'):
    """
    Returns count of unique non-null values in specified column.
    """
    unique_vals = df[col].dropna().unique()
    return len(unique_vals)

n_unique = count_unique_postcodes(d)
print(f"Unique postcodes in d: {n_unique}")

In [ ]:
# Extract first 2 digits of postcode as `admin_unit` id
d['admin_unit'] = d['postcode'].astype(str).str[:2]

# Replace 'na' (from NaN conversion) with actual NaN
d.loc[d['admin_unit'] == 'na', 'admin_unit'] = np.nan

# Enumerate unique admin units
n_unique_admin = count_unique_postcodes(
    d, 
    col = 'admin_unit',
    )
print(f"Unique admin units in d: {n_unique_admin}")
print(f"\nAdmin unit distribution:")
d['admin_unit'].value_counts().sort_index()

In [ ]:
# Map `admin_unit` → `oblast`
### Uses ADMIN_UNIT_TO_OBLAST from mappings.py
### Source: Ukrposhta postal code system: https://en.wikipedia.org/wiki/Postal_codes_in_Ukraine

d['oblast'] = d['admin_unit'].map(ADMIN_UNIT_TO_OBLAST)

# Verify mapping
print(f"Mapped oblasts: {d['oblast'].notna().sum()}/{len(d)}")
print(f"Unmapped admin units: {d[d['oblast'].isna()]['admin_unit'].unique()}")
print(f"\nOblast distribution:")
d['oblast'].value_counts()

#### PRELIM: Encode _raions_ two ways
**Note:** Ukrposhta postcode lookup will not return Russian-occupied districts. These are denoted `<Rus-occupied>` in `raion_postcode`:<br>

|Prefix|Region|$n$ raions|
|------|------|------------------|
|83xxx|Donetsk city|0 (`<Rus-occupied>`)|
|91xxx|Luhansk city|0 (`<Rus-occupied>`)|
|94xxx|Luhansk oblast|0 (`<Rus-occupied>`)|
|95-99|Crimea/Sevastopol|0 (`<Rus-occupied>`)|

In [ ]:
# Map latitude/longitude coordinates → Ukrainian raions via Nominatim API
### Source: ukraine_raion_lookup.py (option 4)
### For Ukraine, Nominatim returns raion in the "district" field
### Rate-limited: 1 req/sec on public instance; not suitable for bulk geocoding

def raion_from_point_nominatim(lat, lon, user_agent, email=None):
    """
    Reverse geocode lat/lon to Ukrainian raion via OSM Nominatim.
    Returns raion name from 'district' field, or None if not found.
    """
    params = {
        'lat': lat,
        'lon': lon,
        'format': 'jsonv2',
        'addressdetails': 1,
    }
    headers = {'User-Agent': user_agent}
    if email:
        params['email'] = email
    
    try:
        resp = requests.get(
            'https://nominatim.openstreetmap.org/reverse',
            params=params,
            headers=headers,
            timeout=10,
        )
        resp.raise_for_status()
        data = resp.json()
        sleep(NOMINATIM_DELAY_SEC)  # respect rate limit
        return data.get('address', {}).get('district')
    except Exception as e:
        print(f"Error geocoding ({lat}, {lon}): {e}")
        return None

# Apply to dataframe with progress bar
raions_nominatim = []

for idx, row in tqdm(d.iterrows(), total=len(d), desc="Geocoding raions"):
    raion = raion_from_point_nominatim(
        row['latitude'], 
        row['longitude'], 
        user_agent=NOMINATIM_USER_AGENT,
    )
    raions_nominatim.append(raion)

d['raion_nominatim_ua'] = raions_nominatim

print(f"\nRaion geocoding complete!")
print(f"Raions found: {d['raion_nominatim_ua'].notna().sum()}/{len(d)}")
print(f"\nSample results:")
d[['latitude', 'longitude', 'raion_nominatim_ua']].head(10)

In [ ]:
# Map Ukrainian raion names → English translations
### Uses RAION_UA_TO_EN from mappings.py
### Source: https://en.wikipedia.org/wiki/Raions_of_Ukraine (post-2020 reform: 136 raions)

d['raion_nominatim_en'] = d['raion_nominatim_ua'].map(RAION_UA_TO_EN)

# Report coverage
mapped = d['raion_nominatim_en'].notna().sum()
total_with_ua = d['raion_nominatim_ua'].notna().sum()
print(f"Mapped to English: {mapped}/{total_with_ua}")

# Show any unmapped Ukrainian raions for dictionary updates
unmapped = d[d['raion_nominatim_ua'].notna() & d['raion_nominatim_en'].isna()]['raion_nominatim_ua'].unique()
if len(unmapped) > 0:
    print(f"\nUnmapped raions (add to mappings.py):")
    for r in unmapped:
        print(f"    '{r}': '',")

print(f"\nSample results:")
d[['raion_nominatim_ua', 'raion_nominatim_en']].head(10)

In [ ]:
# Extract postcode directory from .7z archive
import py7zr

POSTCODE_7Z_PATH = f'{DATA_RAW}/{POSTCODE_7Z}'

# Extract if .7z exists
if os.path.exists(POSTCODE_7Z_PATH):
    # Check if already extracted by looking for any CSV with Ukrainian postal keywords
    existing_csvs = [f for f in os.listdir(DATA_RAW) if f.endswith('.csv') and 'індекс' in f.lower()]
    
    if not existing_csvs:
        print(f"Extracting {POSTCODE_7Z_PATH}...")
        with py7zr.SevenZipFile(POSTCODE_7Z_PATH, mode='r') as archive:
            archive.extractall(path=DATA_RAW)
        print(f"Extracted to: {DATA_RAW}/")
        existing_csvs = [f for f in os.listdir(DATA_RAW) if f.endswith('.csv') and 'індекс' in f.lower()]
    
    # Set path to the extracted CSV
    if existing_csvs:
        POSTCODE_DIR_PATH = f'{DATA_RAW}/{existing_csvs[0]}'
        print(f"Postcode directory: {POSTCODE_DIR_PATH}")
    else:
        # Fallback: find any newly created CSV
        all_csvs = [f for f in os.listdir(DATA_RAW) if f.endswith('.csv')]
        print(f"CSV files found: {all_csvs}")
        if all_csvs:
            POSTCODE_DIR_PATH = f'{DATA_RAW}/{all_csvs[0]}'
            print(f"Using: {POSTCODE_DIR_PATH}")
else:
    print(f"Archive not found: {POSTCODE_7Z_PATH}")
    print("Download from: https://data.gov.ua/dataset/post-index-and-braches")
    POSTCODE_DIR_PATH = None

In [ ]:
# Map postcodes → Ukrainian raions via Ministry of Community and Territorial Development of Ukraine open data
### Source: ukraine_raion_lookup.py (option 3)
### Data: https://data.gov.ua/dataset/post-index-and-braches
### Note: Column headers are in Ukrainian and may vary between releases; inspect after download

def load_postcode_directory(csv_path):
    """
    Load the data.gov.ua "post-index-and-braches" CSV.
    Tries multiple encodings common for Ukrainian government data.
    Uses semicolon delimiter (European CSV format).
    """
    encodings = ['cp1251', 'windows-1251', 'utf-8', 'iso-8859-5', 'utf-16']
    
    for encoding in encodings:
        try:
            df = pd.read_csv(
                csv_path, 
                encoding=encoding, 
                dtype=str,
                sep=';',            ### European .CSV uses semicolon
                on_bad_lines='skip' ### Skips malformed rows
            )
            print(f"Successfully loaded with encoding: {encoding}")
            return df
        except (UnicodeDecodeError, UnicodeError):
            continue
    
    raise ValueError(f"Could not decode {csv_path} with any known encoding")

def raion_from_postcode(postcode, directory, postcode_col, raion_col):
    """
    Direct table lookup for postcode → raion.
    Ukrainian postal codes do NOT cleanly encode raion in digit positions;
    use this authoritative directory rather than parsing the string.
    """
    row = directory.loc[directory[postcode_col] == str(postcode)]
    if row.empty:
        return None
    return row.iloc[0][raion_col]

# Load directory and inspect columns
try:
    postcode_dir = load_postcode_directory(POSTCODE_DIR_PATH)
    print(f"Postcode directory loaded: {len(postcode_dir):,} entries")
    print(f"Columns: {postcode_dir.columns.tolist()}")
    
    # Use English column names from the file
    # Adjust these if your file has different column names
    POSTCODE_COL = 'Postindex VPZ'    ### postcode column
    RAION_COL = 'Distinct (Rayon)'    ### raion column (note: "Distinct" is likely a typo for "District")
    
    # Verify columns exist
    if POSTCODE_COL not in postcode_dir.columns or RAION_COL not in postcode_dir.columns:
        print(f"\nWARNING: Expected columns not found!")
        print(f"Looking for: '{POSTCODE_COL}', '{RAION_COL}'")
        print(f"Available: {postcode_dir.columns.tolist()}")
    else:
        # Apply lookup to dataframe
        d['raion_postcode'] = d['postcode'].apply(
            lambda pc: raion_from_postcode(pc, postcode_dir, POSTCODE_COL, RAION_COL)
        )
        
        # Replace None with <Rus-occupied> (postcodes in occupied territories not in Ukrposhta directory)
        d['raion_postcode'] = d['raion_postcode'].fillna('<Rus-occupied>')
        
        print(f"\nRaions found via postcode: {(d['raion_postcode'] != '<Rus-occupied>').sum()}/{len(d)}")
        print(f"Rus-occupied: {(d['raion_postcode'] == '<Rus-occupied>').sum()}/{len(d)}")
        print(f"\nSample results:")
        d[['postcode', 'raion_postcode']].head(10)
    
except FileNotFoundError:
    print(f"Postcode directory not found at: {POSTCODE_DIR_PATH}")
    print("Download from: https://data.gov.ua/dataset/post-index-and-braches")
    print("Save as 'postindex.7z' in data/raw/ and run extraction cell above")

In [ ]:
#############################################################################################
# DEBUG: Investigate postcode lookup misses

test_postcode = '83054'
print(f"Investigating postcode: {test_postcode}\n")

# Check both postcode columns in directory
postcode_cols = ['Postindex VPZ', 'Postindex Locality']
for col in postcode_cols:
    if col in postcode_dir.columns:
        exact = postcode_dir[postcode_dir[col] == test_postcode]
        partial = postcode_dir[postcode_dir[col].str.contains(test_postcode, na=False)]
        print(f"'{col}':")
        print(f"  Exact match: {len(exact)} rows")
        print(f"  Partial match: {len(partial)} rows")
        print(f"  Sample values: {postcode_dir[col].dropna().unique()[:10].tolist()}\n")

# Check what postcodes ARE in directory for Donetsk oblast (83-87)
donetsk_postcodes = postcode_dir[postcode_dir['Postindex VPZ'].str.startswith('83', na=False)]
print(f"Donetsk (83xxx) postcodes in directory: {len(donetsk_postcodes)}")
if len(donetsk_postcodes) > 0:
    print(f"Sample: {donetsk_postcodes['Postindex VPZ'].unique()[:10].tolist()}")

# Check Luhansk (91-94)
luhansk_postcodes = postcode_dir[postcode_dir['Postindex VPZ'].str.startswith('92', na=False)]
print(f"\nLuhansk (92xxx) postcodes in directory: {len(luhansk_postcodes)}")
if len(luhansk_postcodes) > 0:
    print(f"Sample: {luhansk_postcodes['Postindex VPZ'].unique()[:10].tolist()}")

# Summary: which oblasts are missing?
print(f"\nOblast prefixes in directory:")
postcode_dir['prefix'] = postcode_dir['Postindex VPZ'].str[:2]
print(postcode_dir['prefix'].value_counts().sort_index())
#############################################################################################

In [ ]:
d.head(5)

In [ ]:
# Validation: Compare raion mappings from both methods

# Check where both methods returned a result
both_valid = d['raion_nominatim_ua'].notna() & (d['raion_postcode'] != '<Rus-occupied>')
n_both = both_valid.sum()

print(f"Rows with both raion values: {n_both}/{len(d)}")

if n_both > 0:
    # Compare results (case-insensitive, strip whitespace)
    d['raion_match'] = d.apply(
        lambda row: (
            str(row['raion_nominatim_ua']).lower().strip() == 
            str(row['raion_postcode']).lower().strip()
        ) if pd.notna(row['raion_nominatim_ua']) and row['raion_postcode'] != '<Rus-occupied>' else np.nan,
        axis=1
    )
    
    n_match = d['raion_match'].sum()
    match_rate = n_match / n_both * 100 if n_both > 0 else 0
    
    print(f"Exact matches: {int(n_match)}/{n_both} ({match_rate:.1f}%)")
    
    # Show mismatches for inspection
    mismatches = d[both_valid & (d['raion_match'] == False)][
        ['postcode', 'latitude', 'longitude', 'raion_nominatim_ua', 'raion_postcode']
    ]
    if len(mismatches) > 0:
        print(f"\nMismatches ({len(mismatches)}):")
        mismatches.head(10)
    else:
        print("\nNo mismatches found!")

In [ ]:
#############################################################################################
# DEBUG: Inspect raw Nominatim response for a sample coordinate
import requests

# Use first row with valid coordinates
sample_row = d[d['latitude'].notna()].iloc[0]
lat, lon = sample_row['latitude'], sample_row['longitude']

print(f"Testing coordinates: ({lat}, {lon})")

params = {
    'lat': lat,
    'lon': lon,
    'format': 'jsonv2',
    'addressdetails': 1,
    }
headers = {'User-Agent': NOMINATIM_USER_AGENT}

resp = requests.get(
    'https://nominatim.openstreetmap.org/reverse',
    params=params,
    headers=headers,
    timeout=10,
    )
data = resp.json()

print(f"\nFull address breakdown:")
for key, value in data.get('address', {}).items():
    print(f"  {key}: {value}")

print(f"\nDisplay name: {data.get('display_name', 'N/A')}")
#############################################################################################

In [ ]:
#Export event-level data prior to aggregation
d.to_csv(f'{DATA_PROC}/d_lvl2_event.csv', index = False) 

### 3. Collapse / export: Bellingcat OSINT Civilian Harm in Ukraine

In [ ]:
# Aggregate at raion level

### Collapses event-level data to raion-level counts
### Uses raion_nominatim_en as primary raion identifier

# Define all dummy variables to aggregate
area_type_vars = ['admn', 'cmrc', 'cltr', 'food', 'hlth', 'hmnt', 
                  'indl', 'mltr', 'rlgs', 'rsdn', 'trsp', 'schl', 'undf']

weapon_sys_vars = ['arst', 'amsl', 'blsc', 'clst', 'crsm', 'mrtr',
                   'rckt', 'tube', 'incd', 'mine', 'ltrm', 'none',
                   'smll', 'thrm', 'unkn', 'vhcl']

all_dummy_vars = area_type_vars + weapon_sys_vars

# Build aggregation dict: sum all dummy vars, count events
agg_dict = {var: 'sum' for var in all_dummy_vars}
agg_dict['id'] = 'count'  # count events per raion

# Aggregate by raion (English name)
d_lvl2_raion = d.groupby('raion_nominatim_en', as_index=False).agg(agg_dict)

# Rename id count column
d_lvl2_raion = d_lvl2_raion.rename(columns={'id': 'n_events'})

# Add oblast mapping (most common oblast per raion)
oblast_map = d.groupby('raion_nominatim_en')['oblast'].agg(
    lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else np.nan
)
d_lvl2_raion['oblast'] = d_lvl2_raion['raion_nominatim_en'].map(oblast_map)

# Reorder columns: raion, oblast, n_events, area types, weapon systems
col_order = ['raion_nominatim_en', 'oblast', 'n_events'] + area_type_vars + weapon_sys_vars
d_lvl2_raion = d_lvl2_raion[col_order]

# Sort by total events (descending)
d_lvl2_raion = d_lvl2_raion.sort_values('n_events', ascending=False).reset_index(drop=True)

# Summary
print(f"Raion-level aggregation: {len(d_lvl2_raion)} raions")
print(f"Total events: {d_lvl2_raion['n_events'].sum():,}")
print(f"\nTop 10 raions by event count:")
d_lvl2_raion.head(10)

In [ ]:
#Export raion-level data for 1:n merge
d.to_csv(f'{DATA_PROC}/d_lvl2_raion.csv', index = False) 

In [ ]:
# Aggregate at oblast level

### Collapses event-level data to oblast-level counts

# Build aggregation dict: sum all dummy vars, count events
agg_dict_obl = {var: 'sum' for var in all_dummy_vars}
agg_dict_obl['id'] = 'count'

# Aggregate by oblast
d_lvl2_oblast = d.groupby('oblast', as_index=False).agg(agg_dict_obl)

# Rename id count column
d_lvl2_oblast = d_lvl2_oblast.rename(columns={'id': 'n_events'})

# Add raion count per oblast
raion_counts = d.groupby('oblast')['raion_nominatim_en'].nunique()
d_lvl2_oblast['n_raions'] = d_lvl2_oblast['oblast'].map(raion_counts)

# Reorder columns: oblast, n_raions, n_events, area types, weapon systems
col_order_obl = ['oblast', 'n_raions', 'n_events'] + area_type_vars + weapon_sys_vars
d_lvl2_oblast = d_lvl2_oblast[col_order_obl]

# Sort by total events (descending)
d_lvl2_oblast = d_lvl2_oblast.sort_values('n_events', ascending=False).reset_index(drop=True)

# Summary
print(f"Oblast-level aggregation: {len(d_lvl2_oblast)} oblasts")
print(f"Total events: {d_lvl2_oblast['n_events'].sum():,}")
print(f"\nOblast counts:")
d_lvl2_oblast

In [ ]:
#Export oblast-level data for (potential) 1:n merge
d.to_csv(f'{DATA_PROC}/d_lvl2_oblast.csv', index = False) 